# Notebook 4 — K-Means Clustering

**DATA5322 — Statistical Machine Learning II**  
Seattle University, Spring 2026  
Team: Ruman Sidhu, Paul Skentzos, Hamda Hassan

---

> **Status:** Placeholder — to be completed after PCA (Notebook 2).

## Planned Content
- Run K-Means on PCA-reduced expression data
- Elbow plot (inertia vs k) and silhouette analysis to validate k=4
- Visualize discovered clusters in PC1/PC2 space
- Characterize clusters by mean expression profile
- Compare to known breast cancer subtypes: Luminal A, Luminal B, HER2-enriched, Triple-Negative

---
## 1. Imports and Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Match setup from previous notebooks
pd.set_option('display.float_format', '{:.4f}'.format)
plt.rcParams['figure.dpi'] = 120
sns.set_theme(style='whitegrid')

DATA_DIR = Path('../data')
PLOTS_DIR = Path('../plots')
PLOTS_DIR.mkdir(exist_ok=True)

print('Setup complete.')

---
## 2. Load PCA-Reduced Data

We load the output from Notebook 2:

- `X_pca_reduced.npy`: the PCA-reduced expression matrix with 529 tumor samples and 24 retained principal components
- `sample_ids.csv`: TCGA sample identifiers

Using the PCA-reduced matrix allows K-Means to cluster tumors based on the strongest expression patterns while avoiding the noise and high dimensionality of the original 5,000-gene matrix.

In [ ]:
X = np.load(DATA_DIR / 'X_pca_reduced.npy')
sample_ids = pd.read_csv(DATA_DIR / 'sample_ids.csv')['SampleID'].tolist()

print(f'PCA-reduced matrix shape : {X.shape}  (samples x principal components)')
print(f'Number of sample IDs     : {len(sample_ids)}')
print(f'Matrix mean              : {X.mean():.6f}')
print(f'Matrix std               : {X.std():.6f}')

---
## 3. Determine Optimal Number of Clusters

We evaluate several values of k using both:

- Elbow Method (K-Means inertia)
- Silhouette Score

The elbow method measures within-cluster compactness, while the silhouette score evaluates how well-separated clusters are.

Because breast cancer is commonly grouped into four major molecular subtypes (Luminal A, Luminal B, HER2-enriched, and Triple-Negative), we expect k≈4 to be a reasonable solution.

In [ ]:
# Evaluate candidate cluster counts

k_values = range(2, 11)

inertias = []
sil_scores = []

for k in k_values:
    km = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=20
    )

    labels = km.fit_predict(X)

    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X, labels))

print('Evaluation complete.')

In [ ]:
results = pd.DataFrame({
    'k': list(k_values),
    'Inertia': inertias,
    'Silhouette Score': sil_scores
})

results

---
## 4. Elbow Method

The elbow method evaluates cluster compactness by plotting inertia versus the number of clusters.

A sharp decrease followed by diminishing returns suggests an appropriate number of clusters.

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(k_values, inertias, marker='o')

plt.xlabel('Number of Clusters (k)')
plt.ylabel('Inertia')
plt.title('Elbow Plot for K-Means')

plt.tight_layout()
plt.show()

---
## 5. Silhouette Analysis

The silhouette score measures how well samples fit within their assigned cluster compared to neighboring clusters.

Higher values indicate better cluster separation.

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(k_values, sil_scores, marker='o')

plt.xlabel('Number of Clusters (k)')
plt.ylabel('Silhouette Score')
plt.title('Silhouette Scores Across Cluster Counts')

plt.tight_layout()
plt.show()

---
### Choosing k

The elbow plot shows a noticeable reduction in inertia up to k=4, after which improvements become more gradual.

Although the silhouette score is highest for k=2, biological studies of breast cancer commonly identify four major molecular subtypes (Luminal A, Luminal B, HER2-enriched, and Triple-Negative). Therefore, k=4 provides a reasonable balance between statistical evidence and biological interpretability.

We proceed with k=4 for downstream analysis.

---
## 6. Final K-Means Clustering

We fit a final K-Means model using k=4 and assign each tumor sample to a cluster.

In [ ]:
from sklearn.cluster import KMeans

K_FINAL = 4

kmeans = KMeans(
    n_clusters=K_FINAL,
    random_state=42,
    n_init=20
)

cluster_labels = kmeans.fit_predict(X)

print("Clustering complete.")
print()
print("Cluster sizes:")

pd.Series(cluster_labels).value_counts().sort_index()

---
## 7. Visualizing Clusters in PC Space

To visualize the discovered structure, we project samples onto the first two principal components and color points by their assigned cluster.

If meaningful biological subtypes exist, we expect at least partial separation between clusters in PC1/PC2 space.

In [ ]:
plt.figure(figsize=(9,6))

for c in sorted(np.unique(cluster_labels)):
    mask = cluster_labels == c

    plt.scatter(
        X[mask, 0],
        X[mask, 1],
        label=f'Cluster {c}',
        alpha=0.7
    )

plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('K-Means Clusters in PCA Space (k=4)')
plt.legend()

plt.tight_layout()
plt.show()

---
### Interpretation of Cluster Structure

The PC1/PC2 visualization shows that K-Means successfully identified four major groups within the tumor samples.

Cluster 2 is clearly separated along PC1, while Clusters 1 and 3 are primarily separated along PC2. Cluster 0 occupies an intermediate region and overlaps somewhat with neighboring groups, suggesting that some tumors share characteristics across molecular subtypes.

Although the clusters are not perfectly separated, the overall structure supports the existence of multiple biologically distinct groups within the dataset. This result is consistent with the known heterogeneity of breast cancer and motivates further comparison with established molecular subtype labels.

---
## 8. Cluster Centroids

Cluster centroids represent the average location of each cluster in principal component space.

Examining centroid positions helps summarize the major differences between clusters.

In [ ]:
centroids = pd.DataFrame(
    kmeans.cluster_centers_,
    columns=[f'PC{i+1}' for i in range(X.shape[1])]
)

centroids.iloc[:, :5]

---
## 9. Save Cluster Assignments

We save cluster labels for downstream analysis and comparison with known molecular subtypes.

In [ ]:
cluster_df = pd.DataFrame({
    'SampleID': sample_ids,
    'Cluster': cluster_labels
})

cluster_df.to_csv(DATA_DIR / 'kmeans_clusters.csv', index=False)

print("Saved:")
print(DATA_DIR / 'kmeans_clusters.csv')
print(f'{len(cluster_df)} samples assigned to {K_FINAL} clusters')

---
# 10. Summary

| Property | Value |
|-----------|---------|
| Input matrix | PCA-reduced BC-TCGA tumor data |
| Samples | 529 |
| Principal Components | 24 |
| Clustering Method | K-Means |
| Candidate k values | 2–10 |
| Selected k | 4 |
| Validation Metrics | Elbow + Silhouette |
| Output File | data/kmeans_clusters.csv |

## Key Takeaways

1. K-Means clustering was performed on the PCA-reduced expression matrix containing 529 tumor samples and 24 retained principal components.

2. The elbow plot showed diminishing returns beyond approximately k=4, supporting the use of four clusters.

3. Although the highest silhouette score occurred at k=2, k=4 was selected because it aligns with known breast cancer molecular subtype structure.

4. The PC1/PC2 visualization revealed four reasonably distinct groups, indicating meaningful biological structure in the data.

5. Cluster centroid analysis showed that PC1 and PC2 are the primary drivers separating the discovered clusters.

6. The resulting cluster assignments can be compared with known subtype labels in future analyses.

In [ ]:
from pathlib import Path

for p in Path("../data").rglob("*"):
    if "subtype" in str(p).lower() or "clinical" in str(p).lower():
        print(p)

In [ ]:
for p in Path("../data").rglob("*.csv"):
    print(p)

In [ ]:
for p in Path("../data").rglob("*.txt"):
    print(p)

In [ ]:
test